# Run programs through QDMI

This is Notebook 4 for the SC26 tutorial.
We will discover the local DDSIM device, compile and submit a program, inspect its results, and reuse the compiled program.

Start with a fresh kernel and run the cells in order.
This notebook uses the same Python 3.15 environment as the rest of the tutorial.

In [ ]:
from collections import Counter

from numpy import cos, pi, sin
from qiskit import QuantumCircuit
from qiskit.visualization import plot_distribution

from mqt.core.mlir import compile_program, submit_program
from mqt.core.qdmi import ProgramFormat, device_ids, open_device

shots = 2048
seed = 17

## Discover a device and inspect its capabilities

We begin by listing the configured devices:

In [ ]:
print("Registered devices:", device_ids())

For this tutorial, we will open the bundled DDSIM simulator:

In [ ]:
device = open_device("mqt.ddsim.default")
print("Device:", device.name())
print("Status:", device.status().name)
formats = device.supported_program_formats()
print("Accepted formats:", [program_format.name for program_format in formats])

The device tells us which program formats it accepts, so the compiler can choose a suitable execution format.

## Compile a program with a predictable distribution

Applying $R_y(\theta)$ to qubit 0 and a controlled-$X$ to qubit 1 prepares $\cos(\theta/2)|00\rangle + \sin(\theta/2)|11\rangle$.

:::{admonition} Your turn
:class: tip

For $\theta=\pi/3$, predict the probabilities of `00` and `11`.
:::

In [ ]:
theta = pi / 3

source = QuantumCircuit(2)
source.ry(theta, 0)
source.cx(0, 1)
source.measure_all()

compiled = compile_program(source, target=device)
print("Selected format:", compiled.program_format.name)
print("Payload size:", len(compiled.payload))

Compilation gives us a `CompiledProgram` containing the serialized program and its target requirements.
Since we did not request a format, the compiler selects OpenQASM 3, its preferred format among those supported by DDSIM.

## Submit, wait, and retrieve counts

Now that the program is ready, we can submit it to the same device and wait for the results:

In [ ]:
job = submit_program(compiled, target=device, num_shots=shots, custom1=seed)
job.wait()
print("Final job status:", job.check().name)
counts = job.get_counts()
print(counts)

`custom1` selects the seed for DDSIM's sampling.
We can now compare the sampled counts with the predicted probabilities:

In [ ]:
expected = {"00": cos(theta / 2) ** 2, "11": sin(theta / 2) ** 2}
plot_distribution([expected, counts], legend=["Analytic probabilities", "Sampled counts"])

:::{dropdown} Explain the result

For $\theta=\pi/3$, the expected probabilities are $3/4$ for `00` and $1/4$ for `11`.
With more shots, the sampled frequencies tend toward those probabilities.
:::

## Recover individual shots

With `get_shots()`, we can retrieve individual outcomes in unspecified order and count them to recover the histogram:

In [ ]:
samples = job.get_shots()
print("First 16 shots:", samples[:16])
print("Recovered counts:", dict(Counter(samples)))

## Reuse the compiled program

We can reuse the compiled program for another job, changing only the number of shots:

In [ ]:
small_job = submit_program(compiled, target=device, num_shots=128, custom1=seed)
small_job.wait()
plot_distribution([small_job.get_counts(), counts], legend=["128 shots", f"{shots} shots"])

:::{admonition} Your turn
:class: tip

Change `theta` to `pi / 2` and predict the new probabilities before rerunning from the source cell.
:::

Changing the circuit means compiling it again, while changing the shot count only requires a new job.

## Checkpoint: Return to the slides

We return to the slides here.
The remaining sections are optional.

## Optional: Inspect the simulator state

DDSIM can also return this program's statevector and probabilities.
By requesting OpenQASM with zero shots, we can inspect the state before measurement:

In [ ]:
state_job = submit_program(source, target=device, program_format=ProgramFormat.QASM3, num_shots=0)
state_job.wait()

state = state_job.get_dense_statevector()
probabilities = state_job.get_dense_probabilities()
print("Statevector:", state)
print("Probabilities:", probabilities)

This is a simulator feature.

## Optional: Change the program format

QIR expresses quantum programs in LLVM IR.
We can choose QIR Base explicitly with `program_format`.
When we pass the source to `submit_program`, it handles compilation and submission in one call:

In [ ]:
qir_job = submit_program(
    source,
    target=device,
    program_format=ProgramFormat.QIR_BASE_MODULE,
    num_shots=shots,
    custom1=seed,
)
qir_job.wait()

qir_counts = qir_job.get_counts()
plot_distribution([counts, qir_counts], legend=[f"Default ({compiled.program_format.name})", "QIR Base"])

Although the format changes, the job workflow and expected probabilities stay the same.

:::{admonition} Read further
:class: seealso

The [QDMI guide](https://mqt.readthedocs.io/projects/core/en/stable/qdmi/index.html) covers device configuration and SDK integrations.
For simulator features and supported programs, see the [DDSIM guide](https://mqt.readthedocs.io/projects/core/en/stable/qdmi/ddsim_device.html).
:::